# date_teim groupby each time, weekday 

```python
import matplotlib.pyplot as plt

df = train_building.copy()

# 시간별 power의 평균 계산
hourly_power = df.groupby(df['date_time'].dt.hour)['power'].mean()

# 요일별 power의 평균 계산
weekly_power = df.groupby(df['date_time'].dt.weekday)['power'].mean()

display(hourly_power)
display(weekly_power)
```

---


```
date_time
0      636.847412
1      586.117059
2      567.134471
3      523.389176
4      557.000471
5      655.912588
6      797.699647
7     1001.745529
8     1541.318824
9     1838.949882
10    1857.193412
...
21    1115.505529
22     844.788706
23     727.500706
Name: power, dtype: float64
date_time
0    1336.897500
1    1323.391875
2    1299.189231
3    1293.974063
4    1314.925000
5    1352.656250
6    1105.532500
Name: power, dtype: float64
```

```python
# 시각화
# figsize=(10, 6): 전체 그림의 크기를 (가로 10인치, 세로 6인치)로 설정
plt.figure(figsize=(10, 6))

# 시간별 power
# subplot(2, 1, 1): 2행 1열 레이아웃에서 첫 번째(위쪽) 축을 활성화
#   - 2: 행(row) 개수
#   - 1: 열(column) 개수
#   - 1: 현재 선택할 축의 위치(index)
plt.subplot(2, 1, 1)

# kind='bar': 막대그래프(bar chart)로 그림
# color='skyblue': 막대 색상을 하늘색으로 지정
hourly_power.plot(kind='bar', color='skyblue')

# title(str): 그래프 제목 문자열 설정
plt.title('Hourly Average Power Consumption')

# xlabel(str): x축 라벨 텍스트 설정
plt.xlabel('Hour of Day')

# ylabel(str): y축 라벨 텍스트 설정
plt.ylabel('Average Power')

# 요일별 power
# subplot(2, 1, 2): 2행 1열 레이아웃에서 두 번째(아래쪽) 축을 활성화
plt.subplot(2, 1, 2)

# kind='bar': 막대그래프로 그림
# color='orange': 막대 색상을 주황색으로 지정
weekly_power.plot(kind='bar', color='orange')

# title(str): 아래 그래프 제목 설정
plt.title('Weekly Average Power Consumption')

# xlabel(str): x축 라벨 설정
#   괄호의 설명은 요일 숫자 매핑(0=월요일, 6=일요일)
plt.xlabel('Day of Week (0=Monday, 6=Sunday)')

# ylabel(str): y축 라벨 설정
plt.ylabel('Average Power')

# tight_layout(): 서브플롯 간 여백을 자동 조정해 제목/라벨 겹침을 방지
plt.tight_layout()

# show(): 현재 Figure를 화면에 렌더링
plt.show()
```

## 시간, 시간대, 주중/주말 파생변수 생성 

```python
# 시간 파생 변수 생성 (0=00시, 23=23시)
df['hour'] = df['date_time'].dt.hour

# 시간대 파생 변수 생성 (아침, 오후, 저녁, 밤)
def get_period_of_day(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['period_of_day'] = df['hour'].apply(get_period_of_day)

# 주중/주말 구분 파생 변수 생성 (1=주말, 0=주중)
df['is_weekend'] = df['date_time'].dt.weekday.apply(lambda x: 1 if x >= 5 else 0)

# 결과 확인
df[['date_time', 'hour', 'period_of_day', 'is_weekend']].head(10)
```

## 다양한 시간 관련 파생 변수 생성 방법 

```python
#월 파생 변수 생성 
train['month'] = train['date_time'].dt.month
#분기 파생 변수 생성
train['quarter'] = train['date_time'].dt.quarter
#연도 파생 변수 생성
train['year'] = train['date_time'].dt.year 

#계절 파생 변수 생성
def get_season(month):
    if month in [3,4,5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Fall'
    else:
        return 'Winter'
train['season'] = train['month'].apply(get_season)
```

## 시간,  시간대, 주중/주말 파생 변수 생성 

- 시간과 요일은 순환적 틍성을 가지고 있습니다. 23시 ~ 0시, 일요일 ~ 월요일, 이러한 순환적 특성을 단순히 정수로 표현할 경우, 모델은 연속적인 시간이라는 사실을 인식하지 못할 수 있다. 
- 이 문제를 해결하기 위해 `Sine`과 `Cosine`변환을 사용 

#### 하루 24시간을 기준으로 시간을 사인과 코사인으로 변환 

- `hour_sin` : 시계의 각도가 0~12로 갈수록 증가, 12~24시로 갈수록 다시 감소하는 패턴 표현 
- `hour_cos` : 시계의 각도가 0~6시로 증가하다가, 6~18까지는 감소한후, 18시에서 다시 증가하는 패턴을 표현 

```python
import pandas as pd

# 시간(hour) 순환적 특성 표현
df['hour_sin'] = np.sin(2 * np.pi * df['date_time'].dt.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['date_time'].dt.hour / 24)

# 요일(day of week) 순환적 특성 표현
df['dayofweek_sin'] = np.sin(2 * np.pi * df['date_time'].dt.weekday / 7)
df['dayofweek_cos'] = np.cos(2 * np.pi * df['date_time'].dt.weekday/ 7)

# 결과 확인
df[['date_time', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos']].head()

```

```

date_time	hour_sin	hour_cos	dayofweek_sin	dayofweek_cos
0	2022-06-01 00:00:00	0.000000	1.000000	0.974928	-0.222521
1	2022-06-01 01:00:00	0.258819	0.965926	0.974928	-0.222521
2	2022-06-01 02:00:00	0.500000	0.866025	0.974928	-0.222521
3	2022-06-01 03:00:00	0.707107	0.707107	0.974928	-0.222521
4	2022-06-01 04:00:00	0.866025	0.500000	0.974928	-0.222521

```

이 코드는 시간(hour)을 사인과 코사인으로 변환해, 모델이 시간의 순환성을 올바르게 학습하도록 만드는 방법입니다.

## 예시 파일
[ 14.1.2.DateTimePandas2.ipynb ](dacon/14.1.2.DateTimePandas2.ipynb)

## 답변
핵심 아이디어는 시간 값을 원 위의 각도로 바꾸는 것입니다.

시간 $h \in \{0,\dots,23\}$를 각도 $\theta$로 변환:
$$
\theta = 2\pi \cdot \frac{h}{24}
$$

그다음 좌표를 만듭니다:
$$
hour\_sin = \sin(\theta), \quad hour\_cos = \cos(\theta)
$$

왜 이렇게 하나요?

1. 순환성 보존  
23시와 0시는 실제로 가깝지만, 정수로 쓰면 23과 0이라 멀어 보입니다.  
사인/코사인으로 바꾸면 원 위에서 두 시각이 이웃이 됩니다.

2. 경계 불연속 문제 해결  
정수 인코딩은 23→0에서 값이 갑자기 점프합니다.  
원형 인코딩은 연속적으로 연결됩니다.

3. 모델 학습에 유리  
선형/트리/신경망 모델이 시간 패턴(예: 새벽 저점, 오후 고점)을 더 자연스럽게 잡습니다.

직관적으로 보면:
- 0시: $(\sin 0, \cos 0) = (0, 1)$
- 6시: $(1, 0)$
- 12시: $(0, -1)$
- 18시: $(-1, 0)$

즉 24시간이 원 한 바퀴로 표현됩니다.  
그리고 sin, cos를 같이 쓰는 이유는 한 값만 쓰면 같은 값이 여러 시간대에서 반복되기 때문입니다(예: sin만 쓰면 6시와 18시를 완전히 구분하기 어려움).

### 추가 자료
- [Feature engineering for cyclical features](https://www.avanwyk.com/encoding-cyclical-features-for-deep-learning/)
- [Scikit-learn preprocessing 가이드](https://scikit-learn.org/stable/modules/preprocessing.html)

## 시계열 데이터 정상성(Stationarity)

- 시계열 데이터가 정상성을 가지려면, 데이터의 통계쩍 특성(예, 평균, 분산, 자기상관)이 지나도 일정하게 유지되어야 한다. 

### 강한 정상성 ( Strict Stationarity )

- 시계열 데이터의 분포가 시간에 따라 변하지 않는 경우 
- 데이터의 평균, 분산 뿐만 아니라 전체 분포가 시간에 따라 일정하게 유지
- 모든 시점에서 동일한 확률 분포를 가진다. 

### 약한 정상성 ( Weak Stationarity )

- 시계열 데이터의 평균, 분산, 그리고 자기 공분산이 시간에 따라 일정하게 유지 
- 시계열 데이터 분석에서 더 자주 사용되는 개념.

1. 평균이 일정
2. 분산이 일정
3. 자기공분산이 시간 차이에만 의존 : 데이터의 자기 공분산이 시간ㄴ의 절대적인 위치에 의존하지 않고, 시간 차이(lag)에만 의존
    - 예를 들어, 오늘의 기온과 어제의 기온 사이의 관계가 7월이든 12월이든 비슷하다면, 이는 자기 공분산이 시간 차이에만 의존한다고 볼 수 있다. 
        - 계절에 상관없이 하루 간격의 기온 변화 패턴이 비슷 

## 비정상성(Non Stationarity) 시계열 데이터 

- 일정하지 않고, 계절적 영향을 받으며, 그래프가 위 또는 아래로 상승 또는 하락하는 경향을 보임 



- 정상성을 가지는 데이터는 예측 모델이 데이터의 일관된 패턴을 더 잘 학습할 수 있게 해준다. 
- 이를 통ㅇ해 예측의 신뢰성을 높일 수 있으며, 데이터의 변동에 따른 예측 오차를 줄이는데 도움이 됨. 

- ARIMA(Autoregressive integrated moving average)와 같은 전통적인 시계열 모델은 데이터가 정상성을 가진다는 전제하에 작동 
- 시계열 데이터가 정상성을 가지면, 데이터의 패턴을 더 쉽게 모델링하고 예측할 수 있다. 
- 이를 확인하기 위해 `ADF(Augmented Dickey-Fuller)`테스트와 같은 통계적 검정 방법이 사용됨. 
- ADF테스트는 데이터가 정상성을 가지는지, 즉 시간에 따른 평균과 분산이 일정한지를 평가하는데 도움을 준다. 
    - 만약 데이터가 비정상성을 가진다면, 이를 정상성으로 변환하기 위해 차분(differencing), 로그 변환(log transformation) 등의 기법이 사용. 

## 정상성 분석 및 ADF 테스트 

```python
import matplotlib.pyplot as plt

daily_data = train_building.copy() 
daily_data = daily_data.set_index('date_time')
daily_data = daily_data.resample('D').mean()


# 시계열 데이터 시각화
plt.figure(figsize=(10, 6))
plt.plot(daily_data['power'], label='Power Consumption')
plt.title('Power Consumption Over Time')
plt.xlabel('Date')
plt.ylabel('Power Consumption')
plt.legend()
plt.show()
```

```python
from statsmodels.tsa.stattools import adfuller

# ADF 테스트 함수
def adf_test(series):
    result = adfuller(series)
    print('ADF Statistic:', result[0])
    print('p-value:', result[1])
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'   {key}: {value}')
        
# ADF 테스트 실행
adf_test(daily_data['power'])
```

---

## result 

```
ADF Statistic: -2.2817378051251858
p-value: 0.17792555308125596
Critical Values:
   1%: -3.512738056978279
   5%: -2.8974898650628984
   10%: -2.585948732897085
```

- ADF Statistic : 검정 통계량, 이값이 음수로 더 작을 수록(즉, 더 큰 음수일수록) 데이터가 정상성을 가질 확률이 높다. 
- p-value : 귀무가설(데이터가 비정상성을 가진다)을 기각할 수 있는지 판단하는 확률 값. 일반적으로 p-value가 0.05보다 작으면 데이터가 정상성을 가진다고 결론 내릴 수 있다. 
- Critical Values : 특정 신뢰 수준(1%, 5%, 10%)에서 임계값을 나타냄. ADF 통계량이 이 임계값보다 작다면, 해당 신뢰 수준에서 귀무가설을 기각할 수 있다. 

해당 데이터는 정상성을 가지지 않으며, 시간에 따라 평균이 나 분산이 변화하고 있음. 
[p-value](./tip/p-value.md)가 0.05보다 크고, Critical Values와 비교해도 ADF 가 큼. 
 

## 차분(Differencing) 특성 생성

```python
# 1차 차분(differencing) 적용
daily_data_diff = daily_data['power'].diff().dropna()

# 차분된 데이터 시각화
plt.figure(figsize=(10, 6))
plt.plot(daily_data_diff, label='Differenced Power Consumption')
plt.title('Differenced Power Consumption Over Time')
plt.xlabel('Date')
plt.ylabel('Differenced Power Consumption')
plt.legend()
plt.show()
```

- [diff function](./tip/diff.md). 

### 차분 후 데이터 ADF 결과

```python
# ADF 테스트 함수
def adf_test(series):
    result = adfuller(series)
    print('ADF Statistic:', result[0])
    print('p-value:', result[1])
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'   {key}: {value}')
```

```
adf_test(daily_data_diff)
ADF Statistic: -7.140385192821452
p-value: 3.3360454655429817e-10
Critical Values:
   1%: -3.526004646825607
   5%: -2.9032002348069774
   10%: -2.5889948363419957

```

# ADF 테스트 한 줄 요약
ADF에서는 보통 귀무가설이 “비정상”이고, p-value가 0.05 이하이면 귀무가설을 기각해서 “정상 시계열일 가능성이 높다”고 봅니다.

즉, **p-value ≤ 0.05면 positive(정상)** 라기보다, 정확히는 **“비정상이라는 귀무가설을 기각했다”**는 뜻입니다.  
ADF에서는 그 결과를 보통 **정상 시계열 쪽 증거가 있다**고 해석합니다.

## ACF 자동 상관 함수와 PACF(부분 자동 상관 함수 )

- ACF ( Autocorrelation Function )
    - 시계열 데이터의 특정 시점과 그 이전의 여러 시점(lag) 간의 상관관계를 측정하는 함수. 
    - 예를들어, 오늘의 값이 어제의 값이나 일주일 전의 값과 얼마나 관련 있는지 측정
    - ACF 그래프에서 x축은 lag이며, y축은 해당 lag에서의 상관 관계 계수를 나타냄. 

- PACF ( Partial Autocorrelation Function )
    - 특정 lag에서 그 이전 시점들의 영향을 제거한 후, 현재 시점과 그 lag간의 상관관계를 측정, 주간에 있는 다른 시간점들의 영향을 배제하고, 순수한 상관관계를 파악할 수 있게 해줌. 